# 🤖 Notebook 04: XGBoost 负荷预测 — 15 分钟粒度

## XGBoost Load Forecasting at 15-Minute Resolution

## 学习目标 (Learning Objectives)
1. 理解梯度提升 (Gradient Boosting) 的核心思想
2. 学会用 XGBoost 做 15 分钟级负荷预测
3. 理解 TimeSeriesSplit 和 gap 参数在 15min 粒度下的含义
4. 掌握模型评估指标 (MAE/RMSE/MAPE)
5. 读懂特征重要性

## 梯度提升 (Gradient Boosting) 原理

### 类比: 一群专家协作

想象你要预测明天下午 2:15 的用电量。你找来了 100 个专家:

**专家 1**: "全年均值大约是 45000MW，我预测 45000MW"
→ 残差 (错误) = 实际值 - 45000

**专家 2**: "让我专修正专家 1 的错误。夏天加 8000，冬天加 3000"
→ 把专家 1 的错误又减少了一些

**专家 3**: "我再修正前两位的错。周末减 10000，工作日下午加 5000"
→ 错误继续减少

...重复 100 轮...

**最终预测 = 专家 1 + 专家 2 + 专家 3 + ... + 专家 100**

核心思想: 每一轮都在学习前一轮的**残差**，逐步逼近真实值。

### 15 分钟粒度的特殊挑战

- **更多数据点**: 每天 96 点，4 个月 = ~11,520 点 → 模型有更多训练样本
- **更细的模式**: 15min 的负荷变化比小时级更平滑，lag 特征更相关
- **gap=96**: TimeSeriesSplit 的 gap 需要设 96 (而不是 24)，防止 lag_24h 跨越训练/测试边界

In [ ]:
from ellectric.pipeline.data_loader import create_loader
from ellectric.pipeline.cleaner import clean_data
from ellectric.pipeline.features import FeatureEngineer
from ellectric.pipeline.forecaster import XGBoostForecaster, persistence_forecast
from ellectric.config import TimeConfig
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = 'notebook'

print(f"TimeConfig: {TimeConfig.points_per_day} 点/天, freq={TimeConfig.freq}")

In [ ]:
# ── 数据准备: 加载 + 清洗 + 特征工程 ────────────
loader = create_loader('shandong')
df = loader.load_data('2024-06-01', '2024-09-30')
df = clean_data(df)

# 特征工程 — Tier 1 (先跑通核心特征建立 baseline)
engineer = FeatureEngineer()
df = engineer.add_tier1_features(df)

feature_cols = engineer.get_feature_columns('tier1')
X = df[feature_cols]
y = df['load_mw']

print(f"═══ 训练数据概览 ═══")
print(f"样本数: {len(X)} 行 (15min 粒度)")
print(f"特征数: {len(feature_cols)} 个")
print(f"特征: {feature_cols}")
print(f"目标: load_mw (直调负荷, MW)")
print(f"时间范围: {df['timestamp'].min()} ~ {df['timestamp'].max()}")
print(f"\n每天数据点数: 96 (15min × 24h)")
print(f"总天数: ~{len(df)//96} 天")

---
## 步骤 1: 训练 XGBoost 模型

### TimeSeriesSplit + gap=96

`train_evaluate()` 内部使用 TimeSeriesSplit 切割数据。
**gap=96** 确保训练和测试之间隔 24 小时（96 个 15min 点），
防止 `lag_24h` 特征跨越训练/测试边界——这叫 look-ahead bias 防护。

```
Fold 1: [训练 0-5760] ... gap=96 ... [测试 5857-...]
Fold 2: [训练 0-7680] ... gap=96 ... [测试 7777-...]
...
```

### 超参数说明

| 参数 | 值 | 含义 |
|------|-----|------|
| `n_estimators` | 100 | 树的数量 (迭代次数) |
| `max_depth` | 6 | 每棵树最大深度 |
| `learning_rate` | 0.1 | 学习率 (每棵树贡献的权重) |
| `n_splits` | 5 | 交叉验证折数 |
| `gap` | 96 | 训练/测试间隔 (96 点 = 24h) |

In [ ]:
# ── 训练 XGBoost ────────────────────────────────
forecaster = XGBoostForecaster(
    n_estimators=100,   # 100 棵树
    max_depth=6,        # 每棵树最大深度
    learning_rate=0.1,  # 学习率
)

# gap=96: 15min 粒度下 96 点 = 24 小时
# 防止 lag_24h 跨越训练/测试边界
result = forecaster.train_evaluate(X, y, n_splits=5, gap=96)

metrics = result['metrics']
print(f"\n═══ XGBoost 评估结果 (Tier 1 特征, 5-fold TS) ═══")
print(f"MAE:   {metrics['mae']:,.0f} MW")
print(f"\n📖 MAE = {metrics['mae']:,.0f} MW 意味着:")
print(f"  平均每个 15min 预测偏差约 {metrics['mae']:,.0f} MW")
avg_load = y.mean()
print(f"  相对平均负荷 ({avg_load:,.0f} MW) 的 {metrics['mae']/avg_load*100:.1f}%")

### 📖 评估指标解读

**MAE (平均绝对误差):**
$$MAE = \frac{1}{n}\sum_{i=1}^{n} |y_i - \hat{y}_i|$$
所有预测偏差绝对值的均值。单位与原始数据相同 (MW)。
→ "平均来说，每个 15 分钟的预测偏差约 ±X MW"

**RMSE (均方根误差):**
$$RMSE = \sqrt{\frac{1}{n}\sum_{i=1}^{n} (y_i - \hat{y}_i)^2}$$
先平方（对大误差惩罚更重）再开方（保持量纲）。
RMSE >= MAE，差距大 = 存在少量极大误差。

**MAPE (平均绝对百分比误差):**
$$MAPE = \frac{100}{n}\sum_{i=1}^{n} \left|\frac{y_i - \hat{y}_i}{y_i}\right|$$
跨数据集比较的标准指标。MAPE=5% 表示预测平均偏离 5%。

**注意:** 15 分钟粒度下 MAE 的物理含义:
- 每小时级数据的 MAE=500MW → "每小时的预测偏差 500MW"
- 15 分钟级数据的 MAE=500MW → "每 15 分钟的预测偏差 500MW"
- 15min MAE 通常比 1h MAE **更低**（因为相邻 15min 的负荷更接近）

In [ ]:
# ── 对比持续法基线 ──────────────────────────────
persist = persistence_forecast(df)
persist_mae = (persist - df['load_mw']).abs().mean()

# 计算 RMSE 和 MAPE
persist_rmse = np.sqrt(((persist - df['load_mw']) ** 2).mean())
xgb_pred = forecaster.predict(X)
xgb_rmse = np.sqrt(((xgb_pred - y.values) ** 2).mean())

# MAPE (避免除零)
mask = y.values > 1
xgb_mape = np.mean(np.abs((y.values[mask] - xgb_pred[mask]) / y.values[mask])) * 100
persist_mape = np.mean(np.abs((df['load_mw'].values[mask] - persist.values[mask]) / df['load_mw'].values[mask])) * 100

print(f"═══ 模型对比 ═══")
print(f"{'指标':<12} {'持续法':>12} {'XGBoost':>12} {'提升':>10}")
print(f"{'─'*48}")
print(f"{'MAE (MW)':<12} {persist_mae:>12,.0f} {metrics['mae']:>12,.0f} {(persist_mae-metrics['mae'])/persist_mae*100:>9.1f}%")
print(f"{'RMSE (MW)':<12} {persist_rmse:>12,.0f} {xgb_rmse:>12,.0f} {(persist_rmse-xgb_rmse)/persist_rmse*100:>9.1f}%")
print(f"{'MAPE (%)':<12} {persist_mape:>11.2f}% {xgb_mape:>11.2f}% {(persist_mape-xgb_mape)/persist_mape*100:>9.1f}%")

improvement = (persist_mae - metrics['mae']) / persist_mae * 100
if improvement > 0:
    print(f"\n✅ XGBoost 比持续法好 {improvement:.1f}% — 模型学到了日周期之外的额外模式")
else:
    print(f"\n⚠ XGBoost 没比持续法好，检查特征工程或超参数")

print(f"\n💡 持续法在 15min 粒度下的 MAE={persist_mae:.0f} MW — 这是 96 步前 (24h) 的负荷偏差。")
print("如果持续法 MAE 很小，说明日周期模式非常强——昨天的同一时刻是最好的预测器。")

---
## 步骤 2: 特征重要性 (Feature Importance)

XGBoost 内置特征重要性，告诉你哪些特征对预测贡献最大。

使用 **gain** 指标: 特征在所有树的所有分割点上的平均信息增益。
值越高 = 这个特征越重要。

In [ ]:
importance = result['feature_importance']
import_df = pd.DataFrame(
    sorted(importance.items(), key=lambda x: x[1], reverse=True),
    columns=['特征', '重要性 (gain)']
)

print("═══ 特征重要性排名 (gain) ═══")
for i, row in import_df.iterrows():
    bar = '█' * int(row['重要性 (gain)'] / import_df['重要性 (gain)'].max() * 25)
    print(f"  {i+1}. {row['特征']:20s} {bar} ({row['重要性 (gain)']:.1f})")

top_feature = import_df.iloc[0]['特征']
print(f"\n📖 最重要的特征是 '{top_feature}'。")
if top_feature == 'lag_24h':
    print("lag_24h 排第一说明: 电力负荷有极强的日自相关性——昨天同时刻是最好的预测起点。")
    print("这验证了持续法的有效性，也说明 15min 粒度的惯性很强。")
elif top_feature == 'hour':
    print("hour 排第一说明: 日内时段是负荷变化的最大驱动因素（早晚高峰 vs 凌晨低谷）。")

# ── 可视化特征重要性 ─────────────────────────────
fig = go.Figure()
fig.add_trace(go.Bar(
    x=import_df['重要性 (gain)'].values,
    y=import_df['特征'].values,
    orientation='h',
    marker=dict(color='#1f77b4'),
    text=[f'{v:.1f}' for v in import_df['重要性 (gain)'].values],
    textposition='outside'
))
fig.update_layout(
    title='XGBoost 特征重要性 (Tier 1)',
    xaxis_title='重要性 (gain)',
    yaxis_title='特征',
    height=350,
    margin=dict(l=120)
)
fig.show()

---
## 步骤 3: 预测 vs 实际 可视化

In [ ]:
# ── 全量预测 + 误差分布 ────────────────────────
all_pred = forecaster.predict(X)

# 选取最后一周 (7天 × 96 = 672 点) 做详细展示
n_show = 672
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    subplot_titles=('实际 vs XGBoost 预测 (最后一周)', '预测误差分布'),
    row_heights=[0.6, 0.4],
    vertical_spacing=0.1
)

# 子图1: 实际 vs 预测 (最后一周)
fig.add_trace(
    go.Scatter(x=df['timestamp'].iloc[-n_show:], y=df['load_mw'].iloc[-n_show:],
               name='实际负荷', mode='lines',
               line=dict(color='#1f77b4', width=1.5)),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=df['timestamp'].iloc[-n_show:], y=all_pred[-n_show:],
               name='XGBoost 预测', mode='lines',
               line=dict(color='#ff7f0e', width=1, dash='dash')),
    row=1, col=1
)

# 子图2: 误差分布直方图
errors = all_pred - df['load_mw'].values
fig.add_trace(
    go.Histogram(x=errors, nbinsx=40, name='预测误差',
                 marker=dict(color='#2ca02c')),
    row=2, col=1
)
fig.add_vline(x=0, line_dash='dash', line_color='red', row=2, col=1)

# 标注统计量
fig.add_annotation(
    xref='x2', yref='y2',
    x=max(errors)*0.6, y=0.9,
    text=f'均值: {errors.mean():.0f} MW<br>标准差: {errors.std():.0f} MW',
    showarrow=False,
    bgcolor='rgba(255,255,255,0.8)',
    row=2, col=1
)

fig.update_layout(
    title=f'XGBoost 15min 负荷预测 (MAE={metrics["mae"]:.0f} MW)',
    hovermode='x unified',
    height=600
)
fig.update_yaxes(title_text='负荷 (MW)', row=1, col=1)
fig.update_yaxes(title_text='频次', row=2, col=1)
fig.update_xaxes(title_text='时间', row=2, col=1)
fig.show()

---
## 步骤 4: 模拟交易 P&L (Profit & Loss)

用简化的电力交易模型比较两种策略的"盈亏"水平:
- **持续法策略**: 按昨天同时刻的负荷"买电"
- **XGBoost 策略**: 按 ML 预测的负荷"买电"

P&L 公式:
$$P\&L_t = -|\hat{y}_t - y_t| \times price$$
偏差越小 → 亏得越少。这是极简化的学习模型，不反映真实电力交易。

In [ ]:
from ellectric.pipeline.forecaster import calculate_pnl

price = 300  # 假设电价 300 元/MWh (山东现货市场近似均价)

# 持续法 P&L
persist_pnl = calculate_pnl(df['load_mw'], persist, price_per_mwh=price)
# XGBoost P&L
xgb_pnl = calculate_pnl(df['load_mw'], pd.Series(all_pred, index=df.index), price_per_mwh=price)

fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    subplot_titles=('负荷预测 vs 实际', '累计模拟交易损益 (P&L)'),
    row_heights=[0.5, 0.5],
    vertical_spacing=0.1
)

# 子图1: 负荷
fig.add_trace(
    go.Scatter(x=df['timestamp'], y=df['load_mw'],
               name='实际负荷', mode='lines',
               line=dict(color='#1f77b4', width=1)),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=df['timestamp'], y=all_pred,
               name='XGBoost 预测', mode='lines',
               line=dict(color='#ff7f0e', width=0.8)),
    row=1, col=1
)

# 子图2: 累计 P&L
fig.add_trace(
    go.Scatter(x=df['timestamp'], y=persist_pnl,
               name='持续法 P&L', mode='lines',
               line=dict(color='#d62728', width=1.5)),
    row=2, col=1
)
fig.add_trace(
    go.Scatter(x=df['timestamp'], y=xgb_pnl,
               name='XGBoost P&L', mode='lines',
               line=dict(color='#2ca02c', width=1.5)),
    row=2, col=1
)

# 标注最终 P&L
final_persist = persist_pnl.iloc[-1]
final_xgb = xgb_pnl.iloc[-1]
fig.add_annotation(
    xref='x2', yref='y2',
    x=df['timestamp'].iloc[len(df)//2], y=min(final_persist, final_xgb)*0.5,
    text=f'持续法: {final_persist:,.0f} 元<br>XGBoost: {final_xgb:,.0f} 元<br>差: {final_xgb-final_persist:,.0f} 元',
    showarrow=False, bgcolor='rgba(255,255,255,0.8)',
    row=2, col=1
)

fig.update_layout(
    title=f'持续法 vs XGBoost — 模拟交易损益 (电价 {price} 元/MWh)',
    hovermode='x unified', height=650
)
fig.update_yaxes(title_text='负荷 (MW)', row=1, col=1)
fig.update_yaxes(title_text='累计损益 (元)', row=2, col=1)
fig.update_xaxes(title_text='时间', row=2, col=1)
fig.show()

print(f"\n═══ 模拟交易总结 ═══")
print(f"持续法累计 P&L: {final_persist:,.0f} 元")
print(f"XGBoost 累计 P&L: {final_xgb:,.0f} 元")
print(f"XGBoost 额外收益: {final_xgb - final_persist:,.0f} 元")
print(f"\n⚠ 注意: 这是极简化的学习模型，真实电力交易有更复杂的结算规则。")

---
## 步骤 5: 挑战 — 尝试 Tier 2/3 特征

现在你已经跑通了 Tier 1 baseline。试着添加更多特征，看看效果是否有提升。

**实验模板:**
1. 添加 Tier 2 → 看 MAE 是否下降
2. 添加 Tier 3 → 看是否有边际提升
3. 对比特征重要性排名的变化

> 💡 **提示 (Hint)**:
> 15 分钟粒度下，lag 特征和滚动统计特征的效果通常比小时级数据更好，
> 因为更密集的采样让短期自相关性更强。

In [ ]:
# ── 实验: Tier 2 特征 ───────────────────────────
df2 = engineer.add_tier2_features(df)  # df 来自之前的 Tier 1
t2_cols = engineer.get_feature_columns('tier2')
X2 = df2[t2_cols]

forecaster2 = XGBoostForecaster(n_estimators=100, max_depth=6, learning_rate=0.1)
result2 = forecaster2.train_evaluate(X2, y, n_splits=5, gap=96)

print(f"═══ Tier 1 vs Tier 2 对比 ═══")
print(f"Tier 1 MAE: {metrics['mae']:,.0f} MW")
print(f"Tier 2 MAE: {result2['metrics']['mae']:,.0f} MW")
improvement2 = (metrics['mae'] - result2['metrics']['mae']) / metrics['mae'] * 100
print(f"提升: {improvement2:+.1f}%")

# ── 实验: Tier 3 特征 ────────────────────────────
df3 = engineer.add_tier3_features(df2)
t3_cols = engineer.get_feature_columns('tier3')
X3 = df3[t3_cols]

forecaster3 = XGBoostForecaster(n_estimators=100, max_depth=6, learning_rate=0.1)
result3 = forecaster3.train_evaluate(X3, y, n_splits=5, gap=96)

print(f"\n═══ Tier 2 vs Tier 3 对比 ═══")
print(f"Tier 2 MAE: {result2['metrics']['mae']:,.0f} MW")
print(f"Tier 3 MAE: {result3['metrics']['mae']:,.0f} MW")
improvement3 = (result2['metrics']['mae'] - result3['metrics']['mae']) / result2['metrics']['mae'] * 100
print(f"提升: {improvement3:+.1f}%")

print(f"\n📖 Tier 3 特征加入了 rolling 统计和 sin/cos 编码。")
print("如果 MAE 下降 = 这些高级特征提供了增量信息。")
print("如果 MAE 持平或上升 = 特征冗余或过拟合——Tier 1 就足够了。")

---
## 📝 学习笔记 (Key Takeaways)

- XGBoost 通过 100 棵树逐步修正错误来逼近真实值
- **gap=96** (15min 粒度) 防止 look-ahead bias——过去预测未来
- 特征重要性告诉你模型的"思考过程"
- **永远和持续法对比**——如果 XGBoost 不比持续法好，回去检查
- 15 分钟粒度让模型有更多训练数据（96 点/天 vs 24 点/天），但也要防止过拟合
- 模拟 P&L 是连接"预测精度"和"交易价值"的桥梁

## 🤔 反思 (Reflection)

1. 如果 MAPE > 15%，可能是什么原因？15min 粒度下 MAPE 预期比小时级高还是低？
2. 在 15min 数据上，`rolling_mean_24h` (96 点) 和 `rolling_std_24h` 比小时数据有什么优势？
3. 如果在 Tier 3 基础上加入风光出力作为特征（`wind_actual_mw`, `solar_actual_mw`），预测效果会变好吗？这是"净负荷预测"的思路。

### 思考题 (Reflection Questions)

1. TimeSeriesSplit 的 `gap=96` 是防什么的？如果设为 `gap=0`，MAE 会怎么变？
2. 在这个 15min 数据集上，预测"明天下午 14:00 的负荷"比预测"下周一 14:00 的负荷"更准。为什么时间越远精度越低？
3. 如果只用 `lag_24h` 一个特征（本质是持续法），XGBoost 的 MAE 应该和持续法接近。试试看！